# Feature Extraction and Selection (FES) Tutorial

This notebook demonstrates two different entrypoints:

- `extract_sequence_features()` -> **feature extraction only**
- `run_feature_extraction_and_selection_pipeline()` -> **extraction + selection + modeling**

It also demonstrates a reproducibility preset:

- `preset="unterlerchner2023"`


In [ ]:
import numpy as np
import pandas as pd

from sequenzo import SequenceData, load_dataset, list_datasets
from sequenzo.feature_extraction_and_selection import (
    extract_sequence_features,
    run_feature_extraction_and_selection_pipeline,
    get_feature_extraction_and_selection_config_preset,
)


## Dataset options

### Option A: built-in datasets (easy start)
Use `list_datasets()` to inspect what is shipped in Sequenzo.

### Option B: public external datasets
You can reproduce paper logic with any open longitudinal state-sequence data.
For Swiss educational trajectories in Unterlerchner et al. (2023), they use TREE data,
which is **not fully open by default** and usually requires request/access approval.


In [ ]:
datasets = list_datasets()
print(datasets)


## Prepare sequence data

This example uses a built-in dataset for a runnable workflow.
Adjust `time_cols` and `states` for your own data.


In [ ]:
df = load_dataset("dyadic_children")
time_cols = sorted([c for c in df.columns if str(c).isdigit()], key=int)

# Keep a smaller sample for quick experimentation
df_small = df.head(300).copy()
states = sorted(pd.unique(df_small[time_cols].values.ravel()))
states = [s for s in states if pd.notna(s)]

seqdata = SequenceData(df_small, time=time_cols, states=states)
print(seqdata)


## 1) Feature extraction only

Use this when you want to inspect engineered trajectory features before selection/modeling.


In [ ]:
features = extract_sequence_features(
    seqdata,
    timing_bin_width=12.0,
    sequencing_max_k=3,
    sequencing_min_support=0.05,
)

print("n_features:", len(features["all_feature_names"]))
features["X_full"].head()


## 2) Full FES pipeline with preset

The preset locks core parameters for stable reproducibility:

- timing bin width: 12 months
- frequent subsequence mining: `max_k=3`, `min_support=0.05`
- Boruta setup: default reproducible settings
- residualization behavior enabled


In [ ]:
# Demo outcome and controls for end-to-end execution.
# Replace with your real outcome and controls.
rng = np.random.default_rng(42)
outcome = rng.normal(loc=5000, scale=1200, size=seqdata.n_sequences)

controls = pd.DataFrame({
    "sex": rng.integers(0, 2, size=seqdata.n_sequences),
    "track": rng.integers(0, 3, size=seqdata.n_sequences),
})

cfg = get_feature_extraction_and_selection_config_preset("unterlerchner2023")
print(cfg)

result = run_feature_extraction_and_selection_pipeline(
    seqdata=seqdata,
    outcome=outcome,
    controls=controls,
    preset="unterlerchner2023",
    problem_type="regression",
    verbose=True,
)

print("selected features:", len(result["selected_feature_names"]))
print("R2:", result.get("r2"))
print("BIC:", result.get("bic"))
result["selected_feature_names"][:20]
